# Visibility Thresholds: The Shape a Thought Needs

## Vector-stroke experiments with CLIPDraw

CLIPDraw turns a text prompt into a 224 × 224 drawing made from coloured Bézier curves. [DiffVG](https://github.com/BachiLi/diffvg) renders the curves differentiably, and [OpenAI CLIP](https://github.com/openai/CLIP) ViT-B/32 supplies the text-image similarity objective used to update their geometry, width, and colour.

QMIND's **Visibility Thresholds** project uses this notebook as its **vector-stroke substrate** to investigate whether high CLIP similarity corresponds to visual concepts that people can genuinely recognize. The notebook produces a final PNG, an editable SVG, and JSON experiment metadata. An optional predictor reports the original CLIPDraw top-five noun labels as an exploratory diagnostic.

**Get started:** open this notebook in Google Colab, select **Runtime → Change runtime type → GPU**, run from top to bottom, and use `smoke_test` before a full experiment. The first setup requires internet access and compiles DiffVG; later setup reruns in the same runtime reuse a matching build.

The optimizer is derived from Kevin Frans' [CLIPDraw repository](https://github.com/kvfrans/clipdraw) and the paper [*CLIPDraw: Exploring Text-to-Drawing Synthesis through Language-Image Encoders*](https://arxiv.org/abs/2106.14843) by Kevin Frans, L. B. Soros, and Olaf Witkowski. This version removes the original video and pixel-optimization utilities while preserving the vector objective and unnormalized-input baseline. See the repository's [third-party notices](THIRD_PARTY_NOTICES.md) for licensing details.
## Experimental CLIP / CLIPAG comparison

Select `GUIDANCE_MODEL="clip"` (default, original OpenAI loading behavior) or `"clipag"` (official adversarially fine-tuned ViT-B/32 via OpenCLIP). Both use the same vector substrate and optimizer settings. The original `clipdraw.ipynb` is maintained separately. Normalization remains **off** by default.

**Smoke tests:** in a Colab GPU runtime, use `RUN_MODE="smoke_test"`, `PROMPT="A drawing of a banana"`, `SEED=0`, prediction disabled, and run top to bottom with each guidance model. CLIPAG installs pinned dependencies without replacing torch/torchvision and downloads a **1.31 GB** training checkpoint. Allow extra RAM/disk space for its optimizer state.

**Paired full runs:** use `RUN_MODE="full"` (256 paths, 1,000 iterations, four views), the same prompt/seed/normalization/runtime environment, and change only the guidance selector. Rerun **configuration through export** for each run; rerunning optimization alone continues an existing scene. Use fresh matching runtimes or release the previous model before switching to avoid excess GPU memory use. Repeat over several paired seeds and prompts. Different encoders' losses are not interchangeable recognizability scores; use a shared evaluation procedure and blinded human assessment.

Exports have unique directories containing final PNG/SVG, initial SVG, settings, and loss history. Download all files before deleting the Colab runtime. The batch workflow requires fresh Colab validation; local checks do not establish GPU compatibility.

CLIPAG: Roy Ganz and Michael Elad, *CLIPAG: Towards Generator-Free Text-to-Image Generation*, WACV 2024. [Official code/loading instructions](https://github.com/royg27/CLIPAG), [checkpoint record](https://zenodo.org/records/10446026), [OpenCLIP (MIT)](https://github.com/mlfoundations/open_clip). Existing CLIPDraw/DiffVG/OpenAI CLIP attribution above applies.

## Single-run and automated comparison workflows

`EXPERIMENT_MODE="single"` keeps the selected `GUIDANCE_MODEL`, prompt, seed, predictor, and existing export workflow. `EXPERIMENT_MODE="comparison"` uses `COMPARISON_PROMPTS` and `COMPARISON_SEEDS`, running both encoders for every combination. The shared `RUN_MODE` controls smoke/full workload for every run. The comparison uses the same renderer and optimization functions as single-run mode.

**Fresh Colab comparison smoke test:** select a GPU, set `EXPERIMENT_MODE="comparison"`, `RUN_MODE="smoke_test"`, `COMPARISON_PROMPTS=["A drawing of a banana"]`, `COMPARISON_SEEDS=[0]`, normalization off, negative prompts off. Run all cells. Sections 4-7 define shared functions; section 8 performs both runs and displays the matrix and summary table. Confirm two successful runs, `pair_initialization="matched"`, and verified CLIPAG metadata (302 tensors and checkpoint SHA-256). Compare downloaded `initial.svg` files/hashes.

**Full batch:** change `RUN_MODE="full"`, use the three default prompts and seeds `[0]` (six runs), and rerun configuration onward. Repeat with additional seeds as needed. Models are loaded sequentially, once per model per batch. Paired initialization is checked before optimizing the second member. Each run exports immediately; summaries retain failed, interrupted, and pending entries. Interrupting during initialization can leave an unverified pair. Fix the cause and start a new batch; automatic resume is not provided.

Each unique `comparison_<UTC timestamp>_<ID>/` directory contains separate run directories plus `comparison_matrix.png` (300 dpi), `comparison_summary.csv`, and `comparison_summary.json`. The JSON includes fixed batch settings and environment; each row records checkpoint identity, paths, timings, and model-specific training loss. Missing outputs stay visible as placeholders. Download the shared files and all run subdirectories before deleting the runtime. Model-specific losses are diagnostic and are never used to rank human recognizability. The matrix only reads exported images; it creates no substitute drawings.

Earlier CLIP/CLIPAG single-model full runs were validated in Colab by the project team. The new batch workflow requires its own Colab smoke and full tests.


## 1. Environment setup and dependencies

Run this cell once per fresh Colab runtime. It installs the Python dependencies, checks out pinned CLIP and DiffVG revisions, applies the validated Python 3.13/CUDA 13 compatibility patches, and compiles DiffVG with GPU support. The first build can take several minutes and requires internet access.

The setup keeps Colab's matched PyTorch/torchvision pair. It records a runtime-specific build signature, so rerunning the cell skips DiffVG compilation when the existing installation still matches.

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

DIFFVG_COMMIT = "85802a71fbcc72d79cb75716eb4da4392fd09532"
CLIP_COMMIT = "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"
PYBIND11_TAG = "v2.13.6"
DIFFVG_DIR = Path("/content/diffvg")
CLIP_DIR = Path("/content/CLIP")

def run(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

def run_build(command, *, cwd, env):
    """Stream compiler output and retain a useful tail if the build fails."""
    command = [str(part) for part in command]
    print("+", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 200:
            output_tail.pop(0)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"DiffVG setup.py build failed with exit code {return_code}.\n"
            "The final 200 output lines are repeated below for diagnosis:\n"
            + "".join(output_tail)
        )

def checkout(url, destination, commit, *, recursive=False):
    if destination.exists() and not (destination / ".git").exists():
        raise RuntimeError(f"{destination} exists but is not a Git checkout. Remove it and rerun.")
    if not destination.exists():
        command = ["git", "clone", "--filter=blob:none"]
        if recursive:
            command.append("--recurse-submodules")
        run([*command, url, destination])
    run(["git", "fetch", "origin", commit, "--depth", "1"], cwd=destination)
    run(["git", "checkout", "--detach", commit], cwd=destination)
    if recursive:
        run(["git", "submodule", "update", "--init", "--recursive"], cwd=destination)
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=destination, text=True).strip()
    if head != commit:
        raise RuntimeError(f"Expected {destination.name} at {commit}, found {head}.")

def replace_validated(path, old, new, description):
    text = path.read_text()
    count = text.count(old)
    if count == 1:
        path.write_text(text.replace(old, new))
        print(f"Patched {description}.")
    elif count == 0 and new in text:
        print(f"Already patched: {description}.")
    else:
        raise RuntimeError(
            f"Could not safely patch {description} in {path}: expected one original occurrence, found {count}."
        )

run([sys.executable, "-m", "pip", "install", "cmake<4", "setuptools",
     "svgpathtools", "cssutils", "scikit-image", "matplotlib",
     "ftfy", "packaging", "regex", "tqdm"])

# pydiffvg eagerly imports these modules from image.py, parse_svg.py, and optimize_svg.py.
required_imports = {
    "numpy": "numpy",
    "skimage": "scikit-image",
    "matplotlib": "matplotlib",
    "cssutils": "cssutils",
    "svgpathtools": "svgpathtools",
}
missing_imports = []
for module_name, package_name in required_imports.items():
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing_imports.append(f"{module_name} (pip package {package_name}): {exc}")
if missing_imports:
    raise RuntimeError(
        "Required DiffVG Python dependencies could not be imported:\n- "
        + "\n- ".join(missing_imports)
    )

try:
    import torch
    import torchvision
except Exception as exc:
    raise RuntimeError("Colab's PyTorch/torchvision pair could not be imported.") from exc
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select Runtime → Change runtime type → GPU and reconnect.")

checkout("https://github.com/openai/CLIP.git", CLIP_DIR, CLIP_COMMIT)
clip_marker = CLIP_DIR / ".qmind-installed-commit"
if not clip_marker.exists() or clip_marker.read_text().strip() != CLIP_COMMIT:
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", CLIP_DIR])
    clip_marker.write_text(CLIP_COMMIT)
else:
    print("Pinned CLIP is already installed.")

# Editable installs may not refresh the current notebook kernel's sys.path.
# Prepend the pinned checkout so this process imports the intended CLIP module immediately.
clip_root = str(CLIP_DIR.resolve())
if clip_root in sys.path:
    sys.path.remove(clip_root)
sys.path.insert(0, clip_root)
loaded_clip = sys.modules.get("clip")
if loaded_clip is not None:
    loaded_path = Path(getattr(loaded_clip, "__file__", "")).resolve()
    if CLIP_DIR.resolve() not in loaded_path.parents:
        for module_name in [name for name in sys.modules if name == "clip" or name.startswith("clip.")]:
            del sys.modules[module_name]
importlib.invalidate_caches()
try:
    clip = importlib.import_module("clip")
except Exception as exc:
    raise RuntimeError(
        f"Pinned CLIP could not be imported from {CLIP_DIR}. sys.path begins with {sys.path[0]!r}."
    ) from exc
clip_file = Path(clip.__file__).resolve()
if CLIP_DIR.resolve() not in clip_file.parents:
    raise RuntimeError(f"Imported the wrong clip module from {clip_file}; expected it under {CLIP_DIR}.")
print("Pinned CLIP import verified:", clip_file)

checkout("https://github.com/BachiLi/diffvg.git", DIFFVG_DIR, DIFFVG_COMMIT, recursive=True)
pybind_dir = DIFFVG_DIR / "pybind11"
run(["git", "fetch", "origin", "tag", PYBIND11_TAG, "--depth", "1"], cwd=pybind_dir)
run(["git", "checkout", "--detach", PYBIND11_TAG], cwd=pybind_dir)

setup_file = DIFFVG_DIR / "setup.py"
cmake_file = DIFFVG_DIR / "CMakeLists.txt"
replace_validated(setup_file, 'tf_spec = importlib.util.find_spec("tensorflow")', "tf_spec = None",
                  "TensorFlow package detection")
replace_validated(cmake_file, "find_package(TensorFlow)", "set(TensorFlow_FOUND FALSE)",
                  "TensorFlow compilation")
replace_validated(cmake_file, "set(CMAKE_CUDA_STANDARD 11)", "set(CMAKE_CUDA_STANDARD 17)",
                  "CUDA language standard")
replace_validated(cmake_file, '"${CUDA_NVCC_FLAGS} -std=c++11"',
                  '"${CUDA_NVCC_FLAGS} -std=c++17"', "NVCC language standard")
replace_validated(cmake_file, "PROPERTY CXX_STANDARD 11", "PROPERTY CXX_STANDARD 17",
                  "C++ language standard")

signature = "|".join([DIFFVG_COMMIT, PYBIND11_TAG,
                       f"python-{sys.version_info.major}.{sys.version_info.minor}",
                       f"torch-{torch.__version__}", f"cuda-{torch.version.cuda}"])
build_marker = DIFFVG_DIR / ".qmind-build-signature"
try:
    import pydiffvg
    diffvg_imports = True
except ModuleNotFoundError as exc:
    if exc.name not in {"pydiffvg", "diffvg"}:
        raise RuntimeError(
            f"pydiffvg is installed but its required module {exc.name!r} is missing. "
            "Review the dependency verification output above."
        ) from exc
    diffvg_imports = False
except (ImportError, OSError):
    # A missing or ABI-incompatible compiled diffvg extension requires a rebuild.
    diffvg_imports = False

if not diffvg_imports or not build_marker.exists() or build_marker.read_text().strip() != signature:
    shutil.rmtree(DIFFVG_DIR / "build", ignore_errors=True)
    build_env = os.environ.copy()
    build_env["DIFFVG_CUDA"] = "1"
    build_env["PYTHONUNBUFFERED"] = "1"
    build_env["SETUPTOOLS_USE_DISTUTILS"] = "local"
    # This pinned revision's pyproject.toml selects an incompatible Poetry backend.
    # Invoke its original setuptools entry point so the custom CMake build_ext runs.
    run_build([sys.executable, "setup.py", "install"], cwd=DIFFVG_DIR, env=build_env)
    build_marker.write_text(signature)
else:
    print("DiffVG already matches this runtime; skipping rebuild.")

importlib.invalidate_caches()
try:
    clip = importlib.import_module("clip")
except Exception as exc:
    raise RuntimeError(f"Pinned CLIP import verification failed for {CLIP_DIR}.") from exc
try:
    pydiffvg = importlib.import_module("pydiffvg")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        f"DiffVG built, but pydiffvg still needs missing module {exc.name!r}."
    ) from exc
except Exception as exc:
    raise RuntimeError(
        "DiffVG built, but pydiffvg import failed. The compiled extension may be ABI-incompatible."
    ) from exc
clip_file = Path(clip.__file__).resolve()
if CLIP_DIR.resolve() not in clip_file.parents:
    raise RuntimeError(f"Imported clip from {clip_file}, not the pinned checkout {CLIP_DIR}.")
print("Setup complete.")
print("CLIP:", clip_file)
print("pydiffvg:", Path(pydiffvg.__file__).resolve())


## 2. Runtime verification

Confirm that CUDA is available and record the Python, PyTorch, torchvision, CUDA, GPU, and pinned source versions used by the experiment. These values are also included in the exported metadata.

In [ ]:
import platform
import torch
import torchvision
import clip
import pydiffvg

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Reconnect to a Colab GPU runtime and rerun setup.")
device = torch.device("cuda")
environment = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pytorch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "clip_commit": CLIP_COMMIT,
    "diffvg_commit": DIFFVG_COMMIT,
    "pybind11_tag": PYBIND11_TAG,
}
for name, value in environment.items():
    print(f"{name:>15}: {value}")

## 3. Experiment configuration

Edit the Colab form below before initializing the model or scene. `full` preserves the original baseline. `smoke_test` temporarily overrides the workload with 16 paths, 5 iterations, and 1 augmentation so a teammate can quickly verify rendering and gradients.

| Parameter | Default | What it controls | Practical guidance |
| --- | --- | --- | --- |
| `RUN_MODE` | `full` | Selects the baseline or smoke-test workload. | Use `smoke_test` on a new runtime; use `full` for research runs. |
| `ENABLE_PREDICTION` | `False` | Runs the optional top-five noun predictor after optimization. | Diagnostic only; it cannot change the drawing. |
| `PROMPT` | `Watercolor painting of an underwater submarine.` | Text target encoded by CLIP. | Replace with the concept or style being studied. |
| `USE_NEGATIVE_PROMPTS` | `False` | Enables both negative-prompt penalties. | Leave off for the original default behavior. |
| `NEGATIVE_PROMPT_1` | `A badly drawn sketch.` | First concept discouraged when negative prompts are enabled. | Ignored unless `USE_NEGATIVE_PROMPTS` is on. |
| `NEGATIVE_PROMPT_2` | `Many ugly, messy drawings.` | Second discouraged concept. | Ignored unless `USE_NEGATIVE_PROMPTS` is on. |
| `NORMALIZE_CLIP_INPUTS` | `False` | Applies CLIP channel normalization after augmentation. | `False` preserves the original CLIPDraw baseline; changing it defines a different preprocessing condition. |
| `SEED` | `0` | Seeds Python, NumPy, and PyTorch random generators. | Reuse a seed for comparable initialization, but do not expect bitwise cross-hardware identity. |
| `NUM_PATHS` | `256` | Number of trainable Bézier strokes. | More paths increase visual capacity, memory use, and rendering cost. |
| `NUM_ITERATIONS` | `1000` | Number of optimizer steps. | More steps take longer and may refine the result. |
| `NUM_AUGMENTATIONS` | `4` | Random views scored per optimizer step. | More views increase compute; the original loss sums rather than averages them. |
| `MAX_STROKE_WIDTH` | `50.0` | Upper clamp for trainable stroke widths. | Keep the baseline value unless stroke width is an experimental variable. |
| `DISPLAY_EVERY` | `10` | Preview interval in optimizer iterations. | Affects display frequency only, not optimization. |

The exported metadata records both requested and effective workload values. In `smoke_test`, only path count, iteration count, and augmentation count are overridden.
### Guidance selection

`GUIDANCE_MODEL="clip"` preserves the pinned OpenAI loader. `"clipag"` uses the absolute `CLIPAG_CHECKPOINT_PATH`. `DOWNLOAD_CLIPAG_CHECKPOINT=True` downloads only if absent; set it to `False` to require an existing file. Existing corrupt files fail validation and are never replaced automatically.

Keep normalization off for the baseline pair. Test normalization on both models as a separate condition. Rerun configuration through export whenever changing settings: stale model, prompt, preprocessing, or workload configuration produces an error.

### Batch controls

Choose `EXPERIMENT_MODE="single"` (default) or `"comparison"`. Edit the Python lists `COMPARISON_PROMPTS` and `COMPARISON_SEEDS` directly; use nonempty unique prompts/seeds. Comparison mode ignores the single-run `GUIDANCE_MODEL`, `PROMPT`, and `SEED`, and disables the noun predictor for the batch. All other scientific settings are shared. Run configuration through the final batch cell after edits. Single mode continues to use sections 4-7 and the optional predictor.


In [ ]:
EXPERIMENT_MODE = "single"  # @param ["single", "comparison"]
COMPARISON_PROMPTS = ["A drawing of a banana", "A drawing of a lighthouse", "A drawing of a bicycle"]
COMPARISON_SEEDS = [0]
GUIDANCE_MODEL = "clip"  # @param ["clip", "clipag"]
CLIPAG_CHECKPOINT_PATH = "/content/checkpoints/CLIPAG_ViTB32.pt"  # @param {type:"string"}
DOWNLOAD_CLIPAG_CHECKPOINT = True  # @param {type:"boolean"}
RUN_MODE = "full"  # @param ["full", "smoke_test"]
ENABLE_PREDICTION = False  # @param {type:"boolean"}
PROMPT = "Watercolor painting of an underwater submarine."  # @param {type:"string"}
USE_NEGATIVE_PROMPTS = False  # @param {type:"boolean"}
NEGATIVE_PROMPT_1 = "A badly drawn sketch."  # @param {type:"string"}
NEGATIVE_PROMPT_2 = "Many ugly, messy drawings."  # @param {type:"string"}
NORMALIZE_CLIP_INPUTS = False  # @param {type:"boolean"}
SEED = 0  # @param {type:"integer"}
NUM_PATHS = 256  # @param {type:"integer"}
NUM_ITERATIONS = 1000  # @param {type:"integer"}
NUM_AUGMENTATIONS = 4  # @param {type:"integer"}
MAX_STROKE_WIDTH = 50.0  # @param {type:"number"}
DISPLAY_EVERY = 10  # @param {type:"integer"}

if EXPERIMENT_MODE not in {"single", "comparison"}:
    raise ValueError('EXPERIMENT_MODE must be "single" or "comparison".')
if EXPERIMENT_MODE == "comparison":
    if (not isinstance(COMPARISON_PROMPTS, list) or not COMPARISON_PROMPTS
            or any(not isinstance(value, str) or not value.strip() for value in COMPARISON_PROMPTS)):
        raise ValueError("COMPARISON_PROMPTS must be a nonempty list of nonempty strings.")
    if (not isinstance(COMPARISON_SEEDS, list) or not COMPARISON_SEEDS
            or any(type(value) is not int or not 0 <= value < 2**32 for value in COMPARISON_SEEDS)):
        raise ValueError("COMPARISON_SEEDS must be a nonempty list of integers in [0, 2**32).")
    if len(set(COMPARISON_PROMPTS)) != len(COMPARISON_PROMPTS) or len(set(COMPARISON_SEEDS)) != len(COMPARISON_SEEDS):
        raise ValueError("Comparison prompts and seeds must not contain duplicates.")
if GUIDANCE_MODEL not in {"clip", "clipag"}:
    raise ValueError('GUIDANCE_MODEL must be "clip" or "clipag".')
if RUN_MODE not in {"full", "smoke_test"}:
    raise ValueError('RUN_MODE must be "full" or "smoke_test".')
if (GUIDANCE_MODEL == "clipag" or EXPERIMENT_MODE == "comparison") and not Path(CLIPAG_CHECKPOINT_PATH).is_absolute():
    raise ValueError("CLIPAG_CHECKPOINT_PATH must be an absolute Colab path.")
if not PROMPT.strip():
    raise ValueError("PROMPT must not be empty.")
for name, value in {"NUM_PATHS": NUM_PATHS, "NUM_ITERATIONS": NUM_ITERATIONS,
                    "NUM_AUGMENTATIONS": NUM_AUGMENTATIONS, "DISPLAY_EVERY": DISPLAY_EVERY}.items():
    if value < 1:
        raise ValueError(f"{name} must be at least 1.")
if MAX_STROKE_WIDTH < 1.0:
    raise ValueError("MAX_STROKE_WIDTH must be at least 1.0.")

effective = {
    "num_paths": 16 if RUN_MODE == "smoke_test" else NUM_PATHS,
    "num_iterations": 5 if RUN_MODE == "smoke_test" else NUM_ITERATIONS,
    "num_augmentations": 1 if RUN_MODE == "smoke_test" else NUM_AUGMENTATIONS,
}
print("Effective run configuration:", effective)

## 4. Guidance model and rendering helpers

CLIP retains its original loader. CLIPAG uses OpenCLIP **2.24.0** and timm **1.0.22** with exact constraints protecting Colab's torch/torchvision pair. The existing single-run pins are retained for comparison mode.

CLIPAG follows the authors' released plain `ViT-B-32` inference definition: **GELU**, 224-pixel input, 32-pixel patches, 12 blocks per tower, 512-dimensional embeddings, 77-token context. GELU is explicit and checked. Weights do not identify activation; this matches their released inference code rather than inferring QuickGELU from OpenAI ancestry. Their supplied T2I notebook downloads CLIPAG but selects `pretrained="openai"`; that selection is corrected here.

The official archive contains `state_dict`, `module.` prefixes, and optimizer/scaler state. Check publisher size/MD5 and record SHA-256 before restricted deserialization. A narrowly scoped NumPy scalar allowance handles optimizer metadata. Unwrap weights as OpenCLIP does, load strictly without resizing, and compare every loaded tensor. There is no unsafe-pickle, partial-load, or vanilla fallback.

Both models are frozen in evaluation mode. Text uses `no_grad`; optimization-time image encoding remains differentiable. PIL preprocessing is not applied: the existing tensor augmentation and optional normalization are retained. CLIPAG uses fp32 weights without autocast (the authors' `precision="amp"` construction does not itself enable autocast); standard CLIP retains its original CUDA precision. Metadata records this possible comparison confound.


In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import FileLinks, clear_output, display
from PIL import Image
from torchvision import transforms

OPEN_CLIP_VERSION = "2.24.0"
TIMM_VERSION = "1.0.22"
CLIPAG_URL = "https://zenodo.org/records/10446026/files/CLIPAG_ViTB32.pt?download=1"
CLIPAG_EXPECTED_SIZE = 1310567621
CLIPAG_EXPECTED_MD5 = "f0c75ffafd7b1819c9ae0260cf84f958"

import hashlib
import importlib.metadata
import tempfile
import urllib.request
from collections.abc import Mapping


def file_digests(path):
    md5, sha256 = hashlib.md5(), hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            md5.update(chunk)
            sha256.update(chunk)
    return md5.hexdigest(), sha256.hexdigest()


def install_open_clip():
    # Exact constraints protect DiffVG's existing torch ABI; resolution must fail
    # rather than replacing Colab's matched torch/torchvision pair.
    before = (torch.__version__, torchvision.__version__)
    for module, package, expected in [("open_clip", "open_clip_torch", OPEN_CLIP_VERSION),
                                      ("timm", "timm", TIMM_VERSION)]:
        if module in sys.modules and importlib.metadata.version(package) != expected:
            raise RuntimeError(f"A different {package} is imported. Use a fresh runtime.")
    installed = True
    for package, expected in [("open_clip_torch", OPEN_CLIP_VERSION), ("timm", TIMM_VERSION)]:
        try:
            installed = installed and importlib.metadata.version(package) == expected
        except importlib.metadata.PackageNotFoundError:
            installed = False
    if installed:
        import open_clip
        return open_clip
    with tempfile.TemporaryDirectory() as temporary:
        constraints = Path(temporary) / "torch-constraints.txt"
        constraints.write_text(f"torch=={before[0]}\ntorchvision=={before[1]}\n")
        run([sys.executable, "-m", "pip", "install", "--constraint", constraints,
             f"open_clip_torch=={OPEN_CLIP_VERSION}", f"timm=={TIMM_VERSION}"])
    for package, expected in [("torch", before[0]), ("torchvision", before[1]),
                              ("open_clip_torch", OPEN_CLIP_VERSION), ("timm", TIMM_VERSION)]:
        if importlib.metadata.version(package) != expected:
            raise RuntimeError(f"Unexpected {package} version after installation. Restart runtime.")
    import open_clip
    return open_clip


def verified_clipag_checkpoint():
    path = Path(CLIPAG_CHECKPOINT_PATH)
    if not path.is_absolute():
        raise ValueError("CLIPAG_CHECKPOINT_PATH must be absolute.")
    if not path.exists():
        if not DOWNLOAD_CLIPAG_CHECKPOINT:
            raise FileNotFoundError(f"CLIPAG checkpoint missing: {path}. Enable download or supply the official file.")
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            print("Downloading official CLIPAG checkpoint (1.31 GB):", path)
            urllib.request.urlretrieve(CLIPAG_URL, partial)
            md5, _ = file_digests(partial)
            if partial.stat().st_size != CLIPAG_EXPECTED_SIZE or md5 != CLIPAG_EXPECTED_MD5:
                raise RuntimeError("Downloaded CLIPAG file failed publisher size/MD5 verification.")
            partial.replace(path)
        except Exception as exc:
            partial.unlink(missing_ok=True)
            raise RuntimeError(f"CLIPAG download failed for {path}; no model was loaded.") from exc
    if not path.is_file():
        raise RuntimeError(f"CLIPAG checkpoint path is not a file: {path}")
    md5, sha256 = file_digests(path)
    if path.stat().st_size != CLIPAG_EXPECTED_SIZE or md5 != CLIPAG_EXPECTED_MD5:
        raise RuntimeError(f"Corrupt or unofficial CLIPAG checkpoint: {path}. Size/MD5 mismatch; replace explicitly.")
    return path, {"url": CLIPAG_URL, "path": str(path.resolve()), "bytes": path.stat().st_size,
                  "publisher_md5": md5, "sha256": sha256}


def load_clipag_state(path):
    if not hasattr(torch.serialization, "safe_globals"):
        raise RuntimeError("PyTorch lacks restricted-loading support; use a current Colab runtime.")
    # Official optimizer metadata contains a NumPy float64 scalar. These are the
    # only additional allowed globals, scoped to this verified archive's load.
    numpy_core = np._core if hasattr(np, "_core") else np.core
    numpy_scalar = numpy_core.multiarray.scalar
    allowed = [np.dtype, type(np.dtype("float64")),
               (numpy_scalar, "numpy.core.multiarray.scalar"), numpy_scalar]
    try:
        with torch.serialization.safe_globals(allowed):
            checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    except Exception as exc:
        raise RuntimeError("CLIPAG restricted deserialization failed; no unsafe or vanilla fallback.") from exc
    if not isinstance(checkpoint, Mapping) or "state_dict" not in checkpoint:
        raise RuntimeError("Official CLIPAG format requires a training checkpoint containing state_dict.")
    raw = checkpoint["state_dict"]
    if not isinstance(raw, Mapping) or not raw or not all(
        isinstance(key, str) and key.startswith("module.") and torch.is_tensor(value)
        for key, value in raw.items()
    ):
        raise RuntimeError("CLIPAG state_dict must contain module.-prefixed tensor weights.")
    if len(raw) != 302:
        raise RuntimeError(f"Official CLIPAG ViT-B/32 requires 302 tensors; found {len(raw)}.")
    state = {key[len("module."):]: value for key, value in raw.items()}
    training_name = checkpoint.get("name")
    del checkpoint, raw  # release optimizer/scaler state before model construction
    return state, training_name


def guidance_configuration():
    return {"experiment_mode": EXPERIMENT_MODE, "guidance_model": GUIDANCE_MODEL, "prompt": PROMPT,
            "negative_prompts": [NEGATIVE_PROMPT_1, NEGATIVE_PROMPT_2],
            "use_negative_prompts": USE_NEGATIVE_PROMPTS,
            "normalize_clip_inputs": NORMALIZE_CLIP_INPUTS,
            "checkpoint_path": CLIPAG_CHECKPOINT_PATH, "device": str(device)}


def experiment_configuration():
    return {**guidance_configuration(), "run_mode": RUN_MODE, "seed": SEED,
            "requested": [NUM_PATHS, NUM_ITERATIONS, NUM_AUGMENTATIONS, MAX_STROKE_WIDTH],
            "effective": dict(effective), "enable_prediction": ENABLE_PREDICTION,
            "display_every": DISPLAY_EVERY}


def load_guidance_model():
    global model, tokenize, guidance_metadata, loaded_guidance_configuration, completed_configuration
    # Invalidate the previous successful selection before attempting a new load.
    loaded_guidance_configuration = None
    completed_configuration = None
    if GUIDANCE_MODEL == "clip":
        model, _ = clip.load("ViT-B/32", device=device, jit=False)
        tokenize = clip.tokenize
        guidance_metadata = {"guidance_model": "clip", "backend": "OpenAI CLIP",
                             "architecture": "ViT-B/32", "activation": "QuickGELU",
                             "tokenizer": "clip.tokenize", "clip_commit": CLIP_COMMIT,
                             "precision": "original clip.load CUDA behavior", "checkpoint": None}
    elif GUIDANCE_MODEL == "clipag":
        open_clip = install_open_clip()
        checkpoint_path, checkpoint_metadata = verified_clipag_checkpoint()
        state, training_name = load_clipag_state(checkpoint_path)
        # Same model definition as authors' plain ViT-B-32 local-checkpoint loader.
        # Apply state explicitly to expose strict-load results and avoid auto-resizing.
        model, _, _ = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained=None, precision="amp", device="cpu", force_quick_gelu=False,
        )
        config = open_clip.get_model_config("ViT-B-32")
        if (config["embed_dim"] != 512 or config["vision_cfg"]["image_size"] != 224
                or config["vision_cfg"]["patch_size"] != 32
                or config["vision_cfg"]["layers"] != 12
                or config["text_cfg"]["layers"] != 12
                or config["text_cfg"]["context_length"] != 77
                or config["text_cfg"]["vocab_size"] != 49408):
            raise RuntimeError("Unexpected OpenCLIP ViT-B-32 architecture configuration.")
        activations = [block.mlp.gelu for block in
                       [*model.visual.transformer.resblocks, *model.transformer.resblocks]]
        if len(activations) != 24 or not all(type(act) is torch.nn.GELU for act in activations):
            raise RuntimeError("CLIPAG activation mismatch: authors' plain ViT-B-32 loader requires GELU.")
        try:
            incompatible = model.load_state_dict(state, strict=True)
        except Exception as exc:
            raise RuntimeError("CLIPAG checkpoint incompatible with pinned ViT-B-32; refusing partial loading.") from exc
        if incompatible.missing_keys or incompatible.unexpected_keys:
            raise RuntimeError(f"CLIPAG strict loading failed: {incompatible}")
        actual_state = model.state_dict()
        if set(actual_state) != set(state):
            raise RuntimeError("CLIPAG loaded/checkpoint key sets differ.")
        for key, expected in state.items():
            actual = actual_state[key]
            if (actual.shape != expected.shape or not torch.isfinite(expected).all()
                    or not torch.equal(actual, expected.to(dtype=actual.dtype))):
                raise RuntimeError(f"CLIPAG loaded tensor verification failed: {key}")
        verified_count = len(state)
        fingerprint_keys = ["visual.conv1.weight", "visual.proj", "text_projection"]
        fingerprints = {key: hashlib.sha256(actual_state[key].contiguous().numpy().tobytes()).hexdigest()
                        for key in fingerprint_keys}
        del actual_state, state, actual, expected
        model = model.to(device)
        tokenize = open_clip.get_tokenizer("ViT-B-32")
        guidance_metadata = {"guidance_model": "clipag", "backend": "OpenCLIP",
                             "open_clip_torch": OPEN_CLIP_VERSION, "timm": TIMM_VERSION,
                             "architecture": "ViT-B-32", "activation": "GELU",
                             "activation_basis": "authors' released plain ViT-B-32 inference loader",
                             "tokenizer": "open_clip.get_tokenizer(ViT-B-32)",
                             "precision": "amp construction (fp32 weights); no optimization autocast",
                             "checkpoint": checkpoint_metadata, "training_name": training_name,
                             "strict_load": True, "verified_tensor_count": verified_count,
                             "loaded_tensor_sha256": fingerprints}
    else:
        raise ValueError('GUIDANCE_MODEL must be "clip" or "clipag"; no fallback is available.')

    model.eval()
    model.requires_grad_(False)
    guidance_metadata["parameter_dtypes"] = sorted({str(parameter.dtype) for parameter in model.parameters()})
    guidance_metadata["image_input_dtype"] = "torch.float32"
    guidance_metadata["autocast"] = False


def prepare_guidance():
    global text_features, negative_features_1, negative_features_2
    global loaded_guidance_configuration, loaded_model_identity, loaded_tokenizer_identity
    with torch.no_grad():
        text_features = model.encode_text(tokenize([PROMPT]).to(device))
        negative_features_1 = model.encode_text(tokenize([NEGATIVE_PROMPT_1]).to(device))
        negative_features_2 = model.encode_text(tokenize([NEGATIVE_PROMPT_2]).to(device))
    loaded_guidance_configuration = guidance_configuration()
    loaded_model_identity, loaded_tokenizer_identity = id(model), id(tokenize)


def assert_guidance_current():
    expected_effective = {
        "num_paths": 16 if RUN_MODE == "smoke_test" else NUM_PATHS,
        "num_iterations": 5 if RUN_MODE == "smoke_test" else NUM_ITERATIONS,
        "num_augmentations": 1 if RUN_MODE == "smoke_test" else NUM_AUGMENTATIONS,
    }
    if RUN_MODE not in {"full", "smoke_test"} or effective != expected_effective:
        raise RuntimeError("Stale effective workload. Rerun configuration through export.")
    if (loaded_guidance_configuration != guidance_configuration()
            or id(model) != loaded_model_identity or id(tokenize) != loaded_tokenizer_identity):
        raise RuntimeError("Stale guidance configuration/model/tokenizer. Rerun configuration through export.")
    if model.training or any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("Guidance model must stay frozen and in evaluation mode.")


def assert_scene_current():
    assert_guidance_current()
    if globals().get("initialized_configuration") != experiment_configuration():
        raise RuntimeError("Stale scene/workload configuration. Rerun configuration through export.")


def assert_completed_current():
    assert_scene_current()
    if completed_configuration != experiment_configuration():
        raise RuntimeError("No completed run matches this configuration. Reinitialize and optimize the scene.")


if EXPERIMENT_MODE == "single":
    load_guidance_model()
    prepare_guidance()
    print(json.dumps(guidance_metadata, indent=2))
else:
    print("Comparison helpers ready. The final batch cell performs model loading and all runs.")

augmentation_steps = [
    transforms.RandomPerspective(fill=1, p=1, distortion_scale=0.5),
    transforms.RandomResizedCrop(224, scale=(0.7, 0.9)),
]
if NORMALIZE_CLIP_INPUTS:
    augmentation_steps.append(transforms.Normalize(
        (0.48145466, 0.4578275, 0.40821073),
        (0.26862954, 0.26130258, 0.27577711),
    ))
augment = transforms.Compose(augmentation_steps)

def render_scene(shapes, shape_groups, render_seed):
    scene = pydiffvg.RenderFunction.serialize_scene(224, 224, shapes, shape_groups)
    rgba = pydiffvg.RenderFunction.apply(224, 224, 2, 2, render_seed, None, *scene)
    return rgba[:, :, 3:4] * rgba[:, :, :3] + torch.ones(
        rgba.shape[0], rgba.shape[1], 3, device=pydiffvg.get_device()
    ) * (1 - rgba[:, :, 3:4])

def to_pil(image_hwc):
    array = image_hwc.detach().float().cpu().clamp(0, 1).numpy()
    return Image.fromarray(np.uint8(array * 255))

## 5. Vector scene initialization

Seed the random generators, create the configured number of one-to-three-segment cubic Bézier paths, and attach separate Adam optimizers to their control points, stroke widths, and RGBA colours. The initial image shows the scene before any optimization steps.
This cell defines the shared initializer. It runs immediately in single mode; comparison mode calls it separately for every run, resetting all RNG seeds.


In [ ]:
def initialize_scene():
    global completed_configuration, initialized_configuration, shapes, shape_groups
    global canvas_width, canvas_height, points_vars, stroke_width_vars, color_vars
    global points_optimizer, width_optimizer, color_optimizer, initial_svg
    assert_guidance_current()
    completed_configuration = None
    initialized_configuration = None
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    pydiffvg.set_print_timing(False)
    pydiffvg.set_use_gpu(True)
    pydiffvg.set_device(device)

    canvas_width, canvas_height = 224, 224
    shapes, shape_groups = [], []
    for _ in range(effective["num_paths"]):
        num_segments = random.randint(1, 3)
        num_control_points = torch.zeros(num_segments, dtype=torch.int32) + 2
        p0 = (random.random(), random.random())
        points = [p0]
        for _ in range(num_segments):
            radius = 0.1
            p1 = (p0[0] + radius * (random.random() - 0.5), p0[1] + radius * (random.random() - 0.5))
            p2 = (p1[0] + radius * (random.random() - 0.5), p1[1] + radius * (random.random() - 0.5))
            p3 = (p2[0] + radius * (random.random() - 0.5), p2[1] + radius * (random.random() - 0.5))
            points.extend([p1, p2, p3])
            p0 = p3
        points = torch.tensor(points)
        points[:, 0] *= canvas_width
        points[:, 1] *= canvas_height
        path = pydiffvg.Path(num_control_points=num_control_points, points=points,
                             stroke_width=torch.tensor(1.0), is_closed=False)
        shapes.append(path)
        shape_groups.append(pydiffvg.ShapeGroup(
            shape_ids=torch.tensor([len(shapes) - 1]), fill_color=None,
            stroke_color=torch.tensor([random.random(), random.random(), random.random(), random.random()]),
        ))

    points_vars, stroke_width_vars, color_vars = [], [], []
    for path in shapes:
        path.points.requires_grad = True
        points_vars.append(path.points)
        path.stroke_width.requires_grad = True
        stroke_width_vars.append(path.stroke_width)
    for group in shape_groups:
        group.stroke_color.requires_grad = True
        color_vars.append(group.stroke_color)

    points_optimizer = torch.optim.Adam(points_vars, lr=1.0)
    width_optimizer = torch.optim.Adam(stroke_width_vars, lr=0.1)
    color_optimizer = torch.optim.Adam(color_vars, lr=0.01)
    display(to_pil(render_scene(shapes, shape_groups, 0)))
    # Preserve initialization for paired-run verification without consuming RNG state.
    with tempfile.TemporaryDirectory() as temporary:
        initial_svg_path = Path(temporary) / "initial.svg"
        pydiffvg.save_svg(str(initial_svg_path), canvas_width, canvas_height, shapes, shape_groups)
        initial_svg = initial_svg_path.read_text()
    initialized_configuration = experiment_configuration()

if EXPERIMENT_MODE == "single":
    initialize_scene()


## 6. CLIPDraw optimization

Each iteration renders the curves over white, samples the configured augmented views, and minimizes the original summed cosine-similarity objective. Optional negative prompts add their original weighted penalties. Learning-rate changes and width/colour clamps remain faithful to the baseline.

The preview replaces its previous output instead of appending hundreds of images. Wait for the final completion message before running prediction or export.
This cell defines the shared optimization function without changing its scientific operations. Comparison mode invokes it from the final batch cell.


In [ ]:
def optimize_scene():
    global completed_configuration, loss_history, elapsed_seconds, final_image, num_iterations
    assert_scene_current()
    completed_configuration = None
    loss_history = []
    started = time.perf_counter()
    num_iterations = effective["num_iterations"]
    num_augmentations = effective["num_augmentations"]

    for iteration in range(num_iterations):
        if iteration == int(num_iterations * 0.5):
            for group in points_optimizer.param_groups:
                group["lr"] = 0.4
        if iteration == int(num_iterations * 0.75):
            for group in points_optimizer.param_groups:
                group["lr"] = 0.1

        points_optimizer.zero_grad()
        width_optimizer.zero_grad()
        color_optimizer.zero_grad()
        image_hwc = render_scene(shapes, shape_groups, iteration)
        image_nchw = image_hwc.unsqueeze(0).permute(0, 3, 1, 2)
        image_features = model.encode_image(torch.cat([augment(image_nchw) for _ in range(num_augmentations)]))

        loss = 0
        for index in range(num_augmentations):
            current = image_features[index:index + 1]
            loss -= F.cosine_similarity(text_features, current, dim=1)
            if USE_NEGATIVE_PROMPTS:
                loss += 0.3 * F.cosine_similarity(negative_features_1, current, dim=1)
                loss += 0.3 * F.cosine_similarity(negative_features_2, current, dim=1)

        loss.backward()
        points_optimizer.step()
        width_optimizer.step()
        color_optimizer.step()
        with torch.no_grad():
            for path in shapes:
                path.stroke_width.clamp_(1.0, MAX_STROKE_WIDTH)
            for group in shape_groups:
                group.stroke_color.clamp_(0.0, 1.0)

        loss_value = float(loss.detach().item())
        loss_history.append(loss_value)
        if iteration % DISPLAY_EVERY == 0 or iteration == num_iterations - 1:
            clear_output(wait=True)
            if globals().get("batch_progress"):
                print(batch_progress)
            display(to_pil(image_hwc))
            print(f"Iteration {iteration + 1:,}/{num_iterations:,} | loss {loss_value:.6f}")

    elapsed_seconds = time.perf_counter() - started
    final_image = render_scene(shapes, shape_groups, num_iterations)
    clear_output(wait=True)
    display(to_pil(final_image))
    print(f"Finished {num_iterations:,} iterations in {elapsed_seconds:.1f} seconds.")
    print(f"Final optimization loss: {loss_history[-1]:.6f}")
    completed_configuration = experiment_configuration()

if EXPERIMENT_MODE == "single":
    optimize_scene()


## 7. Optional final-image prediction

When `ENABLE_PREDICTION` is enabled, this separate diagnostic compares one augmented view of the completed drawing with the original 2,343-label CLIPDraw noun vocabulary. Because it runs after optimization, it cannot change the generated scene or loss.

The displayed percentages are a softmax over this fixed candidate set. They are relative CLIP scores—not calibrated confidence probabilities or a measure of human recognition.

In [ ]:
if EXPERIMENT_MODE == "single":
    assert_completed_current()
    if ENABLE_PREDICTION:
        # Verbatim 2,343-entry vocabulary from the original CLIPDraw notebook.
        prediction_labels = 'aardvark abyssinian accelerator accordion account accountant acknowledgment acoustic acrylic act action active activity actor actress adapter addition address adjustment adult advantage advertisement advice afghanistan africa aftermath afternoon aftershave afterthought age agenda agreement air airbus airmail airplane airport airship alarm albatross alcohol algebra algeria alibi alley alligator alloy almanac alphabet alto aluminium aluminum ambulance america amount amusement anatomy anethesiologist anger angle angora animal anime ankle answer ant antarctica anteater antelope anthony anthropology apartment apology apparatus apparel appeal appendix apple appliance approval april aquarius arch archaeology archeology archer architecture area argentina argument aries arithmetic arm armadillo armchair armenian army arrow art ash ashtray asia asparagus asphalt asterisk astronomy athlete atm atom attack attempt attention attic attraction august aunt australia australian author authorisation authority authorization avenue babies baboon baby back backbone bacon badge badger bag bagel bagpipe bail bait baker bakery balance balinese ball balloon bamboo banana band bandana bangladesh bangle banjo bank bankbook banker bar barbara barber barge baritone barometer base baseball basement basin basket basketball bass bassoon bat bath bathroom bathtub battery battle bay beach bead beam bean bear beard beast beat beautician beauty beaver bed bedroom bee beech beef beer beet beetle beggar beginner begonia behavior belgian belief believe bell belt bench bengal beret berry bestseller betty bibliography bicycle bike bill billboard biology biplane birch bird birth birthday bit bite black bladder blade blanket blinker blizzard block blood blouse blow blowgun blue board boat bobcat body bolt bomb bomber bone bongo bonsai book bookcase booklet boot border botany bottle bottom boundary bow bowl bowling box boy bra brace bracket brain brake branch brand brandy brass brazil bread break breakfast breath brian brick bridge british broccoli brochure broker bronze brother brother-in-law brow brown brush bubble bucket budget buffer buffet bugle building bulb bull bulldozer bumper bun burglar burma burn burst bus bush business butane butcher butter button buzzard cabbage cabinet cable cactus cafe cake calculator calculus calendar calf call camel camera camp can canada canadian cancer candle cannon canoe canvas cap capital cappelletti capricorn captain caption car caravan carbon card cardboard cardigan care carnation carol carp carpenter carriage carrot cart cartoon case cast castanet cat catamaran caterpillar cathedral catsup cattle cauliflower cause caution cave c-clamp cd ceiling celery celeste cell cellar cello celsius cement cemetery cent centimeter century ceramic cereal certification chain chair chalk chance change channel character chard charles chauffeur check cheek cheese cheetah chef chemistry cheque cherries cherry chess chest chick chicken chicory chief child children chill chime chimpanzee chin china chinese chive chocolate chord christmas christopher chronometer church cicada cinema circle circulation cirrus citizenship city clam clarinet class claus clave clef clerk click client climb clipper cloakroom clock close closet cloth cloud cloudy clover club clutch coach coal coast coat cobweb cockroach cocktail cocoa cod coffee coil coin coke cold collar college collision colombia colon colony color colt column columnist comb comfort comic comma command commission committee community company comparison competition competitor composer composition computer condition condor cone confirmation conga congo conifer connection consonant continent control cook cooking copper copy copyright cord cork cormorant corn cornet correspondent cost cotton couch cougar cough country course court cousin cover cow cowbell crab crack cracker craftsman crate crawdad crayfish crayon cream creator creature credit creditor creek crib cricket crime criminal crocodile crocus croissant crook crop cross crow crowd crown crush cry cub cuban cucumber cultivator cup cupboard cupcake curler currency current curtain curve cushion custard customer cut cuticle cycle cyclone cylinder cymbal dad daffodil dahlia daisy damage dance dancer danger daniel dash dashboard database date daughter david day dead deadline deal death deborah debt debtor decade december decimal decision decrease dedication deer defense deficit degree delete delivery den denim dentist deodorant department deposit description desert design desire desk dessert destruction detail detective development dew diamond diaphragm dibble dictionary dietician difference digestion digger digital dill dime dimple dinghy dinner dinosaur diploma dipstick direction dirt disadvantage discovery discussion disease disgust dish distance distribution distributor diving division divorced dock doctor dog dogsled doll dollar dolphin domain donald donkey donna door dorothy double doubt downtown dragon dragonfly drain drake drama draw drawbridge drawer dream dredger dress dresser dressing drill drink drive driver driving drizzle drop drug drum dry dryer duck duckling dugout dungeon dust eagle ear earth earthquake ease east edge edger editor editorial education edward eel effect egg eggnog eggplant egypt eight elbow element elephant elizabeth ellipse emery employee employer encyclopedia end enemy energy engine engineer engineering english enquiry entrance environment epoch epoxy equinox equipment era error estimate ethernet ethiopia euphonium europe evening event examination example exchange exclamation exhaust ex-husband existence expansion experience expert explanation ex-wife eye eyebrow eyelash eyeliner face facilities fact factory fahrenheit fairies fall family fan fang farm farmer fat father father-in-law faucet fear feast feather feature february fedelini feedback feeling feet felony female fender ferry ferryboat fertilizer fiber fiberglass fibre fiction field fifth fight fighter file find fine finger fir fire fired fireman fireplace firewall fish fisherman flag flame flare flat flavor flax flesh flight flock flood floor flower flugelhorn flute fly foam fog fold font food foot football footnote force forecast forehead forest forgery fork form format fortnight foundation fountain fowl fox foxglove fragrance frame france freckle freeze freezer freighter french freon friction friday fridge friend frog front frost frown fruit fuel fur furniture galley gallon game gander garage garden garlic gas gasoline gate gateway gauge gazelle gear gearshift geese gemini gender geography geology geometry george geranium german germany ghana ghost giant giraffe girdle girl gladiolus glass glider gliding glockenspiel glove glue goal goat gold goldfish golf gondola gong good-bye goose gore-tex gorilla gosling government governor grade grain gram granddaughter grandfather grandmother grandson grape graphic grass grasshopper gray grease great-grandfather great-grandmother greece greek green grenade grey grill grip ground group grouse growth guarantee guatemalan guide guilty guitar gum gun gym gymnast hacksaw hail hair haircut half-brother half-sister halibut hall hallway hamburger hammer hamster hand handball handicap handle handsaw harbor hardboard hardcover hardhat hardware harmonica harmony harp hat hate hawk head headlight headline health hearing heart heat heaven hedge height helen helicopter helium hell helmet help hemp hen heron herring hexagon hill himalayan hip hippopotamus history hobbies hockey hoe hole holiday home honey hood hook hope horn horse hose hospital hot hour hourglass house hovercraft hub hubcap humidity humor hurricane hyacinth hydrant hydrofoil hydrogen hyena hygienic ice icebreaker icicle icon idea ikebana illegal imprisonment improvement impulse inch income increase index india indonesia industry ink innocent input insect instruction instrument insulation insurance interactive interest internet interviewer intestine invention inventory invoice iran iraq iris iron island israel italian italy jacket jaguar jail jam james january japan japanese jar jasmine jason jaw jeans jeep jeff jelly jellyfish jennifer jet jewel jogging john join joke joseph journey judge judo juice july jumbo jump jumper june jury justice jute kale kamikaze kangaroo karate karen kayak kendo kenneth kenya ketchup kettle kettledrum kevin key keyboard keyboarding kick kidney kilogram kilometer kimberly kiss kitchen kite kitten kitty knee knickers knife knight knot knowledge kohlrabi korean laborer lace ladybug lake lamb lamp lan land landmine language larch lasagna latency latex lathe laugh laundry laura law lawyer layer lead leaf learning leather leek leg legal lemonade lentil leo leopard letter lettuce level libra library license lier lift light lightning lilac lily limit linda line linen link lion lip lipstick liquid liquor lisa list literature litter liver lizard llama loaf loan lobster lock locket locust look loss lotion love low lumber lunch lunchroom lung lunge lute luttuce lycra lynx lyocell lyre lyric macaroni machine macrame magazine magic magician maid mail mailbox mailman makeup malaysia male mall mallet man manager mandolin manicure manx map maple maraca marble march margaret margin maria marimba mark mark market married mary mascara mask mass match math mattock may mayonnaise meal measure meat mechanic medicine meeting melody memory men menu mercury message metal meteorology meter methane mexican mexico mice michael michelle microwave middle mile milk milkshake millennium millimeter millisecond mimosa mind mine minibus mini-skirt minister mint minute mirror missile mist mistake mitten moat modem mole mom monday money monkey month moon morning morocco mosque mosquito mother mother-in-law motion motorboat motorcycle mountain mouse moustache mouth move multi-hop multimedia muscle museum music musician mustard myanmar nail name nancy napkin narcissus nation neck need needle neon nepal nephew nerve nest net network news newsprint newsstand nic nickel niece nigeria night nitrogen node noise noodle north north america north korea norwegian nose note notebook notify novel november number numeric nurse nut nylon oak oatmeal objective oboe observation occupation ocean ocelot octagon octave october octopus odometer offence offer office oil okra olive onion open opera operation ophthalmologist opinion option orange orchestra orchid order organ organisation organization ornament ostrich otter ounce output outrigger oval oven overcoat owl owner ox oxygen oyster package packet page pail pain paint pair pajama pakistan palm pamphlet pan pancake pancreas panda pansy panther panties pantry pants panty pantyhose paper paperback parade parallelogram parcel parent parentheses park parrot parsnip part particle partner partridge party passbook passenger passive pasta paste pastor pastry patch path patient patio patricia paul payment pea peace peak peanut pear pedestrian pediatrician peen peer-to-peer pelican pen penalty pencil pendulum pentagon peony pepper perch perfume period periodical peripheral permission persian person peru pest pet pharmacist pheasant philippines philosophy phone physician piano piccolo pickle picture pie pig pigeon pike pillow pilot pimple pin pine ping pink pint pipe pisces pizza place plain plane planet plant plantation plaster plasterboard plastic plate platinum play playground playroom pleasure plier plot plough plow plywood pocket poet point poison poland police policeman polish politician pollution polo polyester pond popcorn poppy population porch porcupine port porter position possibility postage postbox pot potato poultry pound powder power precipitation preface prepared pressure price priest print printer prison probation process processing produce product production professor profit promotion propane property prose prosecution protest protocol pruner psychiatrist psychology ptarmigan puffin pull puma pump pumpkin punch punishment puppy purchase purple purpose push pvc pyjama pyramid quail quality quart quarter quartz queen question quicksand quiet quill quilt quince quit quiver quotation rabbi rabbit racing radar radiator radio radish raft rail railway rain rainbow raincoat rainstorm rake ramie random range rat rate raven ravioli ray rayon reaction reading reason receipt recess record recorder rectangle red reduction refrigerator refund regret reindeer relation relative religion relish reminder repair replace report representative request resolution respect responsibility rest restaurant result retailer revolve revolver reward rhinoceros rhythm rice richard riddle rifle ring rise risk river riverbed road roadway roast robert robin rock rocket rod roll romania romanian ronald roof room rooster root rose rotate route router rowboat rub rubber rugby rule run russia russian rutabaga ruth sack sagittarius sail sailboat sailor salad salary sale salesman salmon salt sampan samurai sand sandra sandwich santa sarah sardine satin saturday sauce saudi arabia sausage save saw saxophone scale scallion scanner scarecrow scarf scene scent schedule school science scissors scooter scorpio scorpion scraper screen screw screwdriver sea seagull seal seaplane search seashore season seat second secretary secure security seed seeder segment select selection self semicircle semicolon sense sentence separated september servant server session sex shade shadow shake shallot shame shampoo shape share shark sharon shears sheep sheet shelf shell shield shingle ship shirt shock shoe shoemaker shop shorts shoulder shovel show shrimp shrine siamese siberian side sideboard sidecar sidewalk sign signature silica silk silver sing singer single sink sister sister-in-law size skate skiing skill skin skirt sky slash slave sled sleep sleet slice slime slip slipper slope smash smell smile smoke snail snake sneeze snow snowboarding snowflake snowman snowplow snowstorm soap soccer society sociology sock soda sofa softball softdrink software soil soldier son song soprano sort sound soup sousaphone south africa south america south korea soy soybean space spade spaghetti spain spandex spark sparrow spear specialist speedboat sphere sphynx spider spike spinach spleen sponge spoon spot spring sprout spruce spy square squash squid squirrel stage staircase stamp star start starter state statement station statistic steam steel stem step step-aunt step-brother stepdaughter step-daughter step-father step-grandfather step-grandmother stepmother step-mother step-sister stepson step-son step-uncle steven stew stick stinger stitch stock stocking stomach stone stool stop stopsign stopwatch store storm story stove stranger straw stream street streetcar stretch string structure study sturgeon submarine substance subway success sudan suede sugar suggestion suit summer sun sunday sundial sunflower sunshine supermarket supply support surfboard surgeon surname surprise susan sushi swallow swamp swan sweater sweatshirt sweatshop swedish sweets swim swimming swing swiss switch sword swordfish sycamore syria syrup system table tablecloth tabletop tachometer tadpole tail tailor taiwan talk tank tanker tanzania target taste taurus tax taxi taxicab tea teacher teaching team technician teeth television teller temper temperature temple tempo tendency tennis tenor tent territory test text textbook texture thailand theater theory thermometer thing thistle thomas thought thread thrill throat throne thumb thunder thunderstorm thursday ticket tie tiger tights tile timbale time timer timpani tin tip tire titanium title toad toast toe toenail toilet tomato tom-tom ton tongue tooth toothbrush toothpaste top tornado tortellini tortoise touch tower town toy tractor trade traffic trail train tramp transaction transmission transport trapezoid tray treatment tree trial triangle trick trigonometry trip trombone trouble trousers trout trowel truck trumpet trunk t-shirt tsunami tub tuba tuesday tugboat tulip tuna tune turkey turkey turkish turn turnip turnover turret turtle tv twig twilight twine twist typhoon tyvek uganda ukraine ukrainian umbrella uncle underclothes underpants undershirt underwear unit united kingdom unshielded use utensil uzbekistan vacation vacuum valley value van vase vault vegetable vegetarian veil vein velvet venezuela venezuelan verdict vermicelli verse vessel vest veterinarian vibraphone vietnam view vinyl viola violet violin virgo viscose vise vision visitor voice volcano volleyball voyage vulture waiter waitress walk wall wallaby wallet walrus war warm wash washer wasp waste watch watchmaker water waterfall wave wax way wealth weapon weasel weather wedge wednesday weed weeder week weight whale wheel whip whiskey whistle white wholesaler whorl wilderness william willow wind windchime window windscreen windshield wine wing winter wire wish witch withdrawal witness wolf woman women wood wool woolen word work workshop worm wound wrecker wren wrench wrinkle wrist writer xylophone yacht yak yam yard yarn year yellow yew yogurt yoke yugoslavian zebra zephyr zinc zipper zone zoo zoology'.split(" ")
        noun_prompts = ["a drawing of a " + label for label in prediction_labels]

        with torch.no_grad():
            # Preserve the original text prompting and scoring calculation.
            nouns_features = model.encode_text(torch.cat([tokenize(noun_prompts).to(device)]))
            prediction_input = final_image.detach().unsqueeze(0).permute(0, 3, 1, 2)
            prediction_features = model.encode_image(augment(prediction_input))
            image_norm = prediction_features / prediction_features.norm(dim=-1, keepdim=True)
            noun_norm = nouns_features / nouns_features.norm(dim=-1, keepdim=True)
            relative_scores = (100.0 * image_norm @ noun_norm.T).softmax(dim=-1)
            values, indices = relative_scores[0].topk(5)

        print("Top predictions over the original CLIPDraw noun vocabulary:\n")
        print(f'{"Label":<24} {"Relative score":>16}')
        print("-" * 42)
        for value, index in zip(values, indices):
            print(f'{prediction_labels[index]:<24} {100 * value.item():>15.2f}%')
        print(
            "\nThese percentages are a softmax over this fixed candidate vocabulary. "
            "They are relative CLIP scores, not calibrated confidence probabilities."
        )
    else:
        print("Final-image prediction is disabled. Set ENABLE_PREDICTION=True to run it.")

## 8. Results and export

Export the final raster drawing, editable vector scene, and experiment record to `/content/clipdraw_output`:

- `clipdraw_final.png` — convenient for viewing and analysis.
- `clipdraw_final.svg` — editable vector paths for inspection or downstream use.
- `experiment.json` — prompt, settings, source revisions, environment, runtime, and final loss.

Use the links produced by the cell to download the files before disconnecting. Colab's `/content` storage is temporary and is lost when the runtime is deleted.

Each export creates a unique model/seed/mode/normalization/run-ID directory. Download all five files: `clipdraw_final.png`, `clipdraw_final.svg`, `initial.svg`, `experiment.json`, `loss_history.json`. Match initial-SVG hashes across paired runs. The last training loss precedes the final optimizer update and is not a separate evaluation of the exported image. Preserve the notebook/repository revision with your downloaded experiment records.

In comparison mode this cell defines the exporter; the final batch cell writes each run into its unique batch directory. Single-run exports retain their existing location and files.


In [ ]:
def export_run(output_root=None, show=True):
    assert_completed_current()
    from datetime import datetime, timezone
    from uuid import uuid4
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    condition = f"{GUIDANCE_MODEL}_seed{SEED}_{RUN_MODE}_norm{str(NORMALIZE_CLIP_INPUTS).lower()}"
    output_dir = (Path(output_root) if output_root is not None else Path("/content/clipdraw_output")) / f"{condition}_{run_id}"
    output_dir.mkdir(parents=True, exist_ok=False)
    png_path = output_dir / "clipdraw_final.png"
    svg_path = output_dir / "clipdraw_final.svg"
    metadata_path = output_dir / "experiment.json"

    pydiffvg.imwrite(final_image.detach().cpu(), str(png_path), gamma=1.0)
    pydiffvg.save_svg(str(svg_path), canvas_width, canvas_height, shapes, shape_groups)
    experiment = {
        "run_id": run_id,
        "implementation": "QMIND clipdraw_clipag.ipynb selectable guidance v1",
        "runtime_packages": {package: importlib.metadata.version(package)
                             for package in ["numpy", "Pillow", "ftfy", "regex", "tqdm"]},
        "guidance": guidance_metadata,
        "configuration": completed_configuration,
        "experiment_mode": EXPERIMENT_MODE,
        "initial_svg_sha256": hashlib.sha256(initial_svg.encode()).hexdigest(),
        "preprocessing": {
            "canvas_size": [224, 224], "render_samples": [2, 2], "background": "white",
            "perspective": {"fill": 1, "p": 1, "distortion_scale": 0.5},
            "random_resized_crop": {"size": 224, "scale": [0.7, 0.9]},
            "mean": [0.48145466, 0.4578275, 0.40821073],
            "std": [0.26862954, 0.26130258, 0.27577711],
            "normalization_enabled": NORMALIZE_CLIP_INPUTS,
        },
        "project": "Visibility Thresholds: The Shape a Thought Needs",
        "substrate": "vector strokes",
        "run_mode": RUN_MODE,
        "prompt": PROMPT,
        "use_negative_prompts": USE_NEGATIVE_PROMPTS,
        "negative_prompts": [NEGATIVE_PROMPT_1, NEGATIVE_PROMPT_2],
        "normalize_clip_inputs": NORMALIZE_CLIP_INPUTS,
        "seed": SEED,
        "requested": {"num_paths": NUM_PATHS, "num_iterations": NUM_ITERATIONS,
                      "num_augmentations": NUM_AUGMENTATIONS, "max_stroke_width": MAX_STROKE_WIDTH},
        "effective": effective,
        "optimizer": {
            "points_adam_lr": 1.0, "width_adam_lr": 0.1, "color_adam_lr": 0.01,
            "negative_prompt_weight": 0.3,
            "point_lr_schedule": {"50_percent": 0.4, "75_percent": 0.1},
        },
        "environment": environment,
        "elapsed_seconds": elapsed_seconds,
        "final_loss": loss_history[-1],
        "final_loss_definition": "last augmented training loss, before final optimizer update",
        "final_render_seed": num_iterations,
        "loss_reduction": "sum over augmented views",
        "loss_history_file": "loss_history.json",
    }
    (output_dir / "initial.svg").write_text(initial_svg)
    (output_dir / "loss_history.json").write_text(json.dumps(loss_history, indent=2))
    metadata_path.write_text(json.dumps(experiment, indent=2))
    if show:
        display(to_pil(final_image))
        display(FileLinks(str(output_dir)))
        print(json.dumps(experiment, indent=2))
    return {"output_dir": str(output_dir), "png_path": str(png_path),
            "svg_path": str(svg_path), "metadata_path": str(metadata_path), "experiment": experiment}

if EXPERIMENT_MODE == "single":
    single_run_result = export_run()


## 8. Paired comparison batch and presentation
In comparison mode this cell runs both models, persists every run status, checks paired initialization hashes, and exports the matrix and CSV/JSON summaries. Models load sequentially, once per model. The predictor is reserved for single-run mode. Pending/interrupted entries and placeholders identify incomplete batches.


In [ ]:
import csv
import gc
import html
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import HTML


def release_experiment_tensors():
    # All references to previous GPU graphs/parameters must go before loading the
    # next encoder. Functions keep iteration-local tensors out of notebook globals.
    for name in ("model", "tokenize", "text_features", "negative_features_1", "negative_features_2",
                 "shapes", "shape_groups", "points_vars", "stroke_width_vars", "color_vars",
                 "points_optimizer", "width_optimizer", "color_optimizer", "final_image"):
        globals().pop(name, None)
    globals()["completed_configuration"] = None
    globals()["initialized_configuration"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def write_comparison_summary(batch_dir, rows, settings):
    # Persist pending and failed runs too, so an interrupted batch is visibly incomplete.
    for row in rows:
        peer = next(r for r in rows if r["prompt"] == row["prompt"] and r["seed"] == row["seed"]
                    and r["guidance_model"] != row["guidance_model"])
        hashes = (row["initial_scene_sha256"], peer["initial_scene_sha256"])
        row["pair_initialization"] = ("matched" if hashes[0] == hashes[1] else "mismatch") if all(hashes) else "unverified"
    payload = {"batch_settings": settings, "complete": all(r["status"] in {"success", "failed"} for r in rows),
               "all_runs_succeeded": all(r["status"] == "success" for r in rows), "runs": rows}
    json_path = batch_dir / "comparison_summary.json"
    csv_path = batch_dir / "comparison_summary.csv"
    # Atomic replacement prevents half-written summaries if a cell is interrupted.
    temporary = json_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(json_path)
    temporary = csv_path.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows({key: json.dumps(value) if isinstance(value, (dict, list)) else value
                         for key, value in row.items()} for row in rows)
    temporary.replace(csv_path)
    return payload


def render_comparison_matrix(batch_dir, rows, settings, fixture=False):
    import matplotlib.pyplot as plt
    pairs = [(prompt, seed) for prompt in settings["prompts"] for seed in settings["seeds"]]
    fig, axes = plt.subplots(len(pairs), 2, figsize=(9, 3.5 * len(pairs)), squeeze=False)
    fig.patch.set_facecolor("#f4f6f9")
    for index, (prompt, seed) in enumerate(pairs):
        for column, name in enumerate(("clip", "clipag")):
            ax = axes[index, column]
            ax.set_facecolor("white")
            row = next(r for r in rows if (r["prompt"], r["seed"], r["guidance_model"]) == (prompt, seed, name))
            png = Path(row["png_path"]) if row["png_path"] else None
            if row["status"] == "success" and png is not None and png.is_file():
                with Image.open(png) as image:
                    ax.imshow(image.convert("RGB"))
            else:
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                ax.text(0.5, 0.54, "RESULT UNAVAILABLE", ha="center", va="center", fontsize=12, color="#9c3544")
                ax.text(0.5, 0.43, f"Status: {row['status']} (see summary)", ha="center", fontsize=9, color="#596477")
            ax.set_aspect("equal", adjustable="box")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if index == 0:
                ax.set_title("Standard CLIP" if name == "clip" else "CLIPAG", fontsize=15, fontweight="bold", pad=14)
            if column == 0:
                import textwrap
                label = "\n".join(textwrap.wrap(prompt, width=78)) + f"  |  Seed {seed}"
                ax.text(0, -0.10, label, transform=ax.transAxes, fontsize=10, va="top", color="#263347")
    title = "CLIP vs. CLIPAG | Vector-stroke comparison"
    if fixture:
        title = "SYNTHETIC FIXTURE TEST | NOT EXPERIMENT RESULTS"
    workload = settings["effective"]
    subtitle = (f"{workload['num_paths']} paths  |  {workload['num_iterations']} iterations  |  "
                f"{workload['num_augmentations']} views  |  Normalization: {settings['normalize_clip_inputs']}  |  "
                f"Negative prompts: {settings['use_negative_prompts']}")
    fig.suptitle(title, fontsize=17, fontweight="bold", x=0.5, y=0.995, color="#172539")
    fig.text(0.5, 0.995 - 0.27 / fig.get_figheight(), subtitle, ha="center", va="top", fontsize=9, color="#596477")
    fig.text(0.5, 0.008, "Actual exported outputs; missing runs remain visible. Training losses are model-specific, not recognizability scores."
             if not fixture else "Synthetic fixtures used only to validate layout, file paths, and missing-result handling.",
             ha="center", fontsize=8, color="#596477")
    fig.tight_layout(rect=(0.035, 0.035, 0.985, 1 - 0.9 / fig.get_figheight()), h_pad=4, w_pad=2)
    target = batch_dir / "comparison_matrix.png"
    fig.savefig(target, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    display(fig)
    plt.close(fig)
    return target


def display_comparison_table(rows):
    headings = ["Prompt / seed", "Guidance", "Status", "Initial pair", "Time (s)", "Training loss (model-specific)"]
    body = []
    for row in rows:
        loss = row["final_training_loss_model_specific"]
        seconds = row["optimization_seconds"]
        values = [f"{row['prompt']} / {row['seed']}", row["guidance_model"], row["status"],
                  row["pair_initialization"], f"{seconds:.1f}" if seconds is not None else "n/a",
                  f"{loss:.6f}" if loss is not None else "n/a"]
        body.append("<tr>" + "".join(f"<td style='padding:6px 10px;border-bottom:1px solid #ddd'>{html.escape(str(v))}</td>" for v in values) + "</tr>")
    display(HTML("<table><thead><tr>" + "".join(f"<th style='padding:8px'>{h}</th>" for h in headings)
                 + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))
    print("Losses are model-specific diagnostics; this table does not rank recognizability.")


def run_comparison():
    global GUIDANCE_MODEL, PROMPT, SEED, batch_progress
    original = (GUIDANCE_MODEL, PROMPT, SEED)
    batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    batch_dir = Path("/content/clipdraw_output") / f"comparison_{batch_id}"
    batch_dir.mkdir(parents=True, exist_ok=False)
    settings = {"batch_id": batch_id, "experiment_mode": "comparison", "run_mode": RUN_MODE,
                "prompts": list(COMPARISON_PROMPTS), "seeds": list(COMPARISON_SEEDS),
                "effective": dict(effective), "max_stroke_width": MAX_STROKE_WIDTH,
                "timing_definition": "execution_seconds includes text/scene preparation, optimization and export; excludes shared model loading",
                "training_loss_definition": "last augmented pre-update training loss; model-specific, not a recognizability ranking",
                "normalize_clip_inputs": NORMALIZE_CLIP_INPUTS,
                "use_negative_prompts": USE_NEGATIVE_PROMPTS,
                "negative_prompts": [NEGATIVE_PROMPT_1, NEGATIVE_PROMPT_2], "environment": dict(environment)}
    rows = []
    for prompt in settings["prompts"]:
        for seed in settings["seeds"]:
            for name in ("clip", "clipag"):
                rows.append({"prompt": prompt, "seed": seed, "guidance_model": name, "status": "pending",
                             "checkpoint_identity": f"OpenAI ViT-B/32; source {CLIP_COMMIT}" if name == "clip" else "Official CLIPAG ViT-B/32",
                             "checkpoint_sha256": None, "checkpoint_expected_md5": CLIPAG_EXPECTED_MD5 if name == "clipag" else None,
                             "guidance_metadata": None, "num_paths": effective["num_paths"],
                             "num_iterations": effective["num_iterations"], "num_augmentations": effective["num_augmentations"],
                             "normalize_clip_inputs": NORMALIZE_CLIP_INPUTS, "use_negative_prompts": USE_NEGATIVE_PROMPTS,
                             "negative_prompts": [NEGATIVE_PROMPT_1, NEGATIVE_PROMPT_2], "initial_scene_sha256": None,
                             "pair_initialization": "unverified", "execution_seconds": None, "optimization_seconds": None,
                             "final_training_loss_model_specific": None, "png_path": None, "svg_path": None,
                             "metadata_path": None, "error": None})
    write_comparison_summary(batch_dir, rows, settings)
    try:
        for name in ("clip", "clipag"):
            release_experiment_tensors()
            GUIDANCE_MODEL = name
            selected = [row for row in rows if row["guidance_model"] == name]
            print(f"Loading {name.upper()} once for {len(selected)} runs. Batch directory: {batch_dir}")
            try:
                load_guidance_model()
            except Exception as exc:
                message = f"Model loading failed: {type(exc).__name__}: {exc}"
                print(message)
                for row in selected:
                    row.update(status="failed", error=message)
                write_comparison_summary(batch_dir, rows, settings)
                continue
            for row in selected:
                PROMPT, SEED = row["prompt"], row["seed"]
                batch_progress = f"{name.upper()} | {PROMPT} | seed {SEED} | {sum(r['status'] in {'success', 'failed'} for r in rows) + 1}/{len(rows)}"
                print(batch_progress)
                row["status"] = "running"
                row["guidance_metadata"] = json.loads(json.dumps(guidance_metadata))
                row["checkpoint_sha256"] = (guidance_metadata.get("checkpoint") or {}).get("sha256")
                write_comparison_summary(batch_dir, rows, settings)
                wall_started = time.perf_counter()
                try:
                    prepare_guidance()
                    initialize_scene()
                    row["initial_scene_sha256"] = hashlib.sha256(initial_svg.encode()).hexdigest()
                    peer = next(r for r in rows if r["prompt"] == PROMPT and r["seed"] == SEED and r["guidance_model"] != name)
                    if peer["initial_scene_sha256"] and peer["initial_scene_sha256"] != row["initial_scene_sha256"]:
                        raise RuntimeError("Paired initial scene hashes differ; refusing an unmatched comparison.")
                    write_comparison_summary(batch_dir, rows, settings)
                    optimize_scene()
                    if not all(np.isfinite(value) for value in loss_history):
                        raise RuntimeError("Nonfinite training loss; result is incomplete.")
                    result = export_run(batch_dir, show=False)
                    row.update(status="success", optimization_seconds=elapsed_seconds,
                               final_training_loss_model_specific=loss_history[-1],
                               png_path=result["png_path"], svg_path=result["svg_path"], metadata_path=result["metadata_path"])
                except Exception as exc:
                    # Store strings only: tracebacks would retain failed GPU graphs.
                    row.update(status="failed", error=f"{type(exc).__name__}: {exc}")
                    print(f"FAILED: {batch_progress}: {row['error']}")
                finally:
                    row["execution_seconds"] = time.perf_counter() - wall_started
                    write_comparison_summary(batch_dir, rows, settings)
                print(f"{batch_progress}: {row['status']}")
    except KeyboardInterrupt:
        for row in rows:
            if row["status"] == "running":
                row.update(status="interrupted", error="User interrupted the batch.")
        print("Batch interrupted. Pending and interrupted runs are preserved in the summary.")
    finally:
        release_experiment_tensors()
        GUIDANCE_MODEL, PROMPT, SEED = original
        batch_progress = None
        write_comparison_summary(batch_dir, rows, settings)
    render_comparison_matrix(batch_dir, rows, settings)
    display_comparison_table(rows)
    display(FileLinks(str(batch_dir), recursive=True))
    print(f"Batch saved: {batch_dir}. Successful runs: {sum(r['status'] == 'success' for r in rows)}/{len(rows)}")
    return {"batch_dir": str(batch_dir), "settings": settings, "runs": rows}


if EXPERIMENT_MODE == "comparison":
    comparison_result = run_comparison()
else:
    print("Single-run workflow selected. Set EXPERIMENT_MODE='comparison' and rerun configuration onward for a batch.")
